# Data Ingestion

In [120]:
import pandas as pd

df = pd.read_csv("../data_raw/raw_train.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   object 
 2   Location           5847 non-null   object 
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   object 
 6   Transmission       5847 non-null   object 
 7   Owner_Type         5847 non-null   object 
 8   Mileage            5845 non-null   object 
 9   Engine             5811 non-null   object 
 10  Power              5811 non-null   object 
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    object 
 13  Price              5847 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 639.6+ KB


# Data Processing

In [121]:
import numpy as np

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   object 
 2   Location           5847 non-null   object 
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   object 
 6   Transmission       5847 non-null   object 
 7   Owner_Type         5847 non-null   object 
 8   Mileage            5845 non-null   object 
 9   Engine             5811 non-null   object 
 10  Power              5811 non-null   object 
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    object 
 13  Price              5847 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 639.6+ KB


Removing/Replacing missing values

In [122]:
# Remove the column 'Unnamed: 0' because it serves no analytical purpose 
# Remoce the column 'New_Price' because it is 86% missing values
df.drop(columns=['Unnamed: 0', 'New_Price'], inplace=True)

# For missing values in Engine, Power, and Seats replace the values with the mode of cars with the same name
nan_columns = ['Engine', 'Power', 'Seats']
for column in nan_columns:
    df[column] = df[column].fillna(df.groupby('Name')[column].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None))
# Remove rows of values that still remain missing
df.dropna(subset=nan_columns, inplace=True)

# Replace missing values in 'Mileage' with '0.0 kmpl'
df['Mileage'] = df['Mileage'].fillna('0.0 kmpl')

Remove units from attributes

In [123]:
# Extract numeric value and unit from column
df[['Mileage_value', 'Mileage_unit']] = df['Mileage'].str.extract(r'(\d+\.?\d*)\s*(\w+\/?\w*)')
df['Mileage_value'] = df['Mileage_value'].astype(float)

# Density map for the different fuel types
density_map = {
    'Petrol': 0.755,
    'Diesel': 0.845
}

# Standardize all 'Mileage' units to kmpl
df['Mileage_kmpl'] = np.where(
    df['Mileage_unit'] == 'km/kg',
    df['Mileage_value'] * df['Fuel_Type'].map(density_map),
    df['Mileage_value']  # already kmpl
)

# Overwrite 'Mileage' with 'Mileage_kmpl' and remove other columns created
df['Mileage'] = df['Mileage_kmpl']
df.drop(columns=['Mileage_value', 'Mileage_unit', 'Mileage_kmpl'], inplace=True)

# Remove units of remaining columns
num_columns = ['Engine', 'Power']
for col in num_columns:
    df[col] = df[col].str.extract(r'(\d+\.?\d*)').astype(float)

Change categorical variables into numerical one hot encoded value

In [124]:
df = pd.get_dummies(df, columns=['Fuel_Type', 'Transmission'], drop_first=False)

Create Car_Age attribute

In [125]:
from datetime import date

current_year = date.today().year
df['Car_Age'] = current_year - df['Year']

Save Processed data to data_clean

In [127]:
df.to_csv('../data_clean/clean_train.csv', index=False)

# Data Analysis

In [ ]:
# Select
df[['Price', 'Mileage', 'Car_Age', 'Seats', 'Power']]

,Price,Mileage,Car_Age,Seats,Power
0,12.50,19.670,11,5.0,126.20
1,4.50,9.815,15,5.0,88.70
2,6.00,20.770,14,7.0,88.76
3,17.74,15.200,13,5.0,140.80
4,3.50,23.080,13,5.0,63.10
...,...,...,...,...,...
5842,4.75,28.400,12,5.0,74.00
5843,4.00,24.400,11,5.0,71.00
5844,2.90,14.000,14,8.0,112.00
5845,2.65,18.900,13,5.0,67.10


In [131]:
# Filter
df[(df['Fuel_Type_Petrol'] == True) & (df['Price'] < 8.5)]

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Car_Age
1,Honda Jazz V,Chennai,2011,46000,First,9.815,1199.0,88.70,5.0,4.50,False,False,True,False,True,15
9,Honda City 1.5 V AT Sunroof,Kolkata,2012,60000,First,16.800,1497.0,116.30,5.0,4.49,False,False,True,True,False,14
21,Hyundai i20 1.2 Magna,Kolkata,2010,45807,First,18.500,1197.0,80.00,5.0,1.87,False,False,True,False,True,16
22,Volkswagen Vento Petrol Highline AT,Kolkata,2010,33000,First,14.400,1598.0,103.60,5.0,2.85,False,False,True,True,False,16
23,Honda City Corporate Edition,Mumbai,2012,51920,First,16.800,1497.0,116.30,5.0,4.25,False,False,True,False,True,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5831,Maruti Celerio VXI,Bangalore,2015,67600,First,23.100,998.0,67.04,5.0,4.00,False,False,True,False,True,11
5832,Hyundai Getz GLE,Coimbatore,2007,66800,First,15.300,1341.0,83.00,5.0,2.20,False,False,True,False,True,19
5838,Honda Brio 1.2 VX MT,Delhi,2013,33746,First,18.500,1198.0,86.80,5.0,3.20,False,False,True,False,True,13
5839,Skoda Superb 3.6 V6 FSI,Hyderabad,2009,53000,First,0.000,3597.0,262.60,5.0,4.75,False,False,True,True,False,17


In [134]:
# Rename
df = df.rename(columns={
    'Mileage': 'Mileage_kmpl',
    'Engine': 'Engine_CC',
    'Power': 'Power_bhp',
    'Price': 'Price_lahk',
    'Fuel_Type_Diesel': 'Diesel',
    'Fuel_Type_Petrol': 'Petrol',
    'Fuel_Type_Electric': 'Electric',
    'Transmission_Automatic': 'Automatic',
    'Transmission_Manual': 'Manual'
})
df

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage_kmpl,Engine_CC,Power_bhp,Seats,Price_lahk,Diesel,Electric,Petrol,Automatic,Manual,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.670,1582.0,126.20,5.0,12.50,True,False,False,False,True,11
1,Honda Jazz V,Chennai,2011,46000,First,9.815,1199.0,88.70,5.0,4.50,False,False,True,False,True,15
2,Maruti Ertiga VDI,Chennai,2012,87000,First,20.770,1248.0,88.76,7.0,6.00,True,False,False,False,True,14
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.200,1968.0,140.80,5.0,17.74,True,False,False,True,False,13
4,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.080,1461.0,63.10,5.0,3.50,True,False,False,False,True,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5842,Maruti Swift VDI,Delhi,2014,27365,First,28.400,1248.0,74.00,5.0,4.75,True,False,False,False,True,12
5843,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,First,24.400,1120.0,71.00,5.0,4.00,True,False,False,False,True,11
5844,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Second,14.000,2498.0,112.00,8.0,2.90,True,False,False,False,True,14
5845,Maruti Wagon R VXI,Kolkata,2013,46000,First,18.900,998.0,67.10,5.0,2.65,False,False,True,False,True,13


In [142]:
# Mutate
df['Price_per_Seat'] = df['Price_lahk'] / df['Seats']
df

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage_kmpl,Engine_CC,Power_bhp,Seats,Price_lahk,Diesel,Electric,Petrol,Automatic,Manual,Car_Age,Price_per_Seat
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.670,1582.0,126.20,5.0,12.50,True,False,False,False,True,11,2.500000
1,Honda Jazz V,Chennai,2011,46000,First,9.815,1199.0,88.70,5.0,4.50,False,False,True,False,True,15,0.900000
2,Maruti Ertiga VDI,Chennai,2012,87000,First,20.770,1248.0,88.76,7.0,6.00,True,False,False,False,True,14,0.857143
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.200,1968.0,140.80,5.0,17.74,True,False,False,True,False,13,3.548000
4,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.080,1461.0,63.10,5.0,3.50,True,False,False,False,True,13,0.700000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5842,Maruti Swift VDI,Delhi,2014,27365,First,28.400,1248.0,74.00,5.0,4.75,True,False,False,False,True,12,0.950000
5843,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,First,24.400,1120.0,71.00,5.0,4.00,True,False,False,False,True,11,0.800000
5844,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Second,14.000,2498.0,112.00,8.0,2.90,True,False,False,False,True,14,0.362500
5845,Maruti Wagon R VXI,Kolkata,2013,46000,First,18.900,998.0,67.10,5.0,2.65,False,False,True,False,True,13,0.530000


In [143]:
# Arrange
df.sort_values('Price_lahk')

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage_kmpl,Engine_CC,Power_bhp,Seats,Price_lahk,Diesel,Electric,Petrol,Automatic,Manual,Car_Age,Price_per_Seat
1660,Tata Nano Lx,Pune,2011,65000,Second,26.00,624.0,35.0,4.0,0.44,False,False,True,False,True,15,0.1100
3039,Maruti Zen LXI,Jaipur,1998,95150,Third,17.30,993.0,60.0,5.0,0.45,False,False,True,False,True,28,0.0900
2758,Hyundai Getz GLS,Pune,2005,86000,Second,15.30,1341.0,83.0,5.0,0.45,False,False,True,False,True,21,0.0900
1577,Maruti 800 Std BSIII,Jaipur,2004,12000,Second,16.10,796.0,37.0,4.0,0.45,False,False,True,False,True,22,0.1125
2520,Tata Nano Cx,Jaipur,2010,57000,First,26.00,624.0,35.0,4.0,0.50,False,False,True,False,True,16,0.1250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1917,BMW 7 Series 740Li,Coimbatore,2018,28060,First,12.05,2979.0,320.0,5.0,93.67,False,False,True,True,False,8,18.7340
1457,Land Rover Range Rover Sport SE,Kochi,2019,26013,First,12.65,2993.0,255.0,5.0,97.07,True,False,False,True,False,7,19.4140
5752,Jaguar F Type 5.0 V8 S,Hyderabad,2015,8000,First,12.50,5000.0,488.1,2.0,100.00,False,False,True,True,False,11,50.0000
5620,Lamborghini Gallardo Coupe,Delhi,2011,6500,Third,6.40,5204.0,560.0,2.0,120.00,False,False,True,True,False,15,60.0000


In [146]:
# Summarize
df.groupby('Name').agg(
    avg_price=('Price_lahk', 'mean'),
    max_power=('Power_bhp', 'max')
)

,avg_price,max_power
Name,,
Ambassador Classic Nova Diesel,1.350000,35.5
Audi A3 35 TDI Attraction,16.500000,143.0
Audi A3 35 TDI Premium,19.250000,143.0
Audi A3 35 TDI Premium Plus,18.900000,143.0
Audi A3 35 TDI Technology,22.500000,143.0
...,...,...
Volvo XC60 D4 Summum,18.250000,163.0
Volvo XC60 D5,19.433333,215.0
Volvo XC60 D5 Inscription,17.180000,215.0
